<a href="https://colab.research.google.com/github/StathisDevves/Industrial/blob/main/Lowest%20Price%20set%20overall%20Opt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# =========================================================
# LOWEST 8-DAY MINI-MILL OPTIMIZATION
# WITH GAS + SOLAR + PV + STORAGE + THERMAL
# =========================================================
# INPUT FILES
#   1) Electricity_Gas Solar Prices_2025_8760.xlsx
#   2) Design.xlsx
#
# OUTPUT FILE
#   Lowest_8Day_Mini_Mill_with_Gas_Solar_PV_Storage.xlsx
#
# WHAT THIS SCRIPT DOES
#   1. Reads hourly data from the 8760 file
#   2. Finds the 8 consecutive days with the lowest average electricity price
#   3. Uses the corresponding electricity, natural gas, and solar rows
#   4. For each of the 8 days independently:
#        - optimizes the daily start hour of EAF1.1
#        - minimizes total daily electricity cost for the 24-process sequence
#   5. Computes extra columns:
#        - PV Generation
#        - battery / thermal binary decisions
#        - SOC columns
#        - cost columns
#   6. Enforces daily simple rules:
#        - sum(Battery Charge) >= 4
#        - sum(Battery Discharge) <= 3
#        - sum(Thermal Discharge) < 9
#        - SOC_t=1 = SOC_t=24 through constant daily rates
#
# IMPORTANT
#   This script implements the formulas directly.
#   If your Design.xlsx contains additional formulas/parameters,
#   adapt the parameter block below.
# =========================================================

input_prices_file = "Electricity_Gas Solar Prices_2025_8760.xlsx"
input_design_file = "Design.xlsx"  # not strictly required by this script, kept for consistency
output_file = "Lowest_8Day_Mini_Mill_with_Gas_Solar_PV_Storage.xlsx"

# =========================================================
# 1. USER PARAMETERS / MODEL ASSUMPTIONS
# =========================================================
PV_EFFICIENCY = 0.18
PV_AREA_M2 = 66700

BATTERY_ENERGY_MWH = 40.0
BATTERY_EFFICIENCY = 0.92
THERMAL_BUFFER_CAPACITY_MWH_TH = 60.0

BATTERY_INITIAL_SOC = 0.50 * BATTERY_ENERGY_MWH
THERMAL_INITIAL_SOC = 0.50 * THERMAL_BUFFER_CAPACITY_MWH_TH

# Constant daily rates chosen to satisfy cyclic daily SOC
# SOC_t=1 = SOC_t=24 for each day if #charge = #discharge and same rate
RB_FRACTION = 0.10    # 10% of battery capacity per active hour
RTH_FRACTION = 0.08   # 8% of thermal capacity per active hour

SCRAP_RATE_DEFAULT = 0.00

# If natural gas is monetized in the thermal cost:
THERMAL_TO_GAS_CONVERSION = 1.0

# =========================================================
# 2. FIXED PROCESS SEQUENCE (24 process-hours)
# =========================================================
# Loads / recovered heat / heat required based on your latest table.
# Where duplicated rows in the prompt were inconsistent, this script keeps
# the first coherent 24-row sequence.

stage_sequence = [
    ("Step 1", "EAF1.1", 72, 9.0, 0.000000, 1, 0, 0),
    ("Step 1", "EAF1.2", 78, 9.3, 0.000000, 1, 0, 0),
    ("Step 1", "EAF1.3", 80, 9.4, 0.000000, 1, 0, 0),
    ("Step 1", "EAF1.4", 72, 9.0, 0.825104, 1, 0, 0),

    ("Step 2", "Secondary Downstream 1.1", 32, 1.0, 7.219660, 0, 1, 0),
    ("Step 2", "Secondary Downstream 1.2", 26, 0.7, 30.941400, 0, 1, 0),
    ("Step 2", "Secondary Downstream 1.3", 20, 0.4, 22.690360, 0, 1, 0),

    ("Step 3", "SD sequence Blue Colour", 15, 0.15, 1.340794, 0, 1, 0),

    ("Step 4", "Minimum Critical Load 1", 12, 0.0, 1.134518, 0, 0, 12),
    ("Step 4", "Minimum Critical Load 2", 12, 0.0, 1.031380, 0, 0, 12),
    ("Step 4", "Minimum Critical Load 3", 12, 0.0, 0.618828, 0, 0, 12),
    ("Step 4", "Minimum Critical Load 4", 12, 0.0, 0.618828, 0, 0, 12),
    ("Step 4", "Minimum Critical Load 5", 12, 0.0, 0.618828, 0, 0, 12),
    ("Step 4", "Minimum Critical Load 6", 12, 0.0, 2.887864, 0, 0, 12),

    ("Step 5", "Preparation Blue 1", 15, 0.15, 2.062760, 0, 1, 0),
    ("Step 5", "Preparation Blue 2", 24, 0.60, 0.515690, 0, 1, 0),
    ("Step 5", "Preparation Blue 3", 38, 1.30, 4.125520, 0, 1, 0),

    ("Step 6", "EAF2.1", 72, 9.0, 0.000000, 1, 0, 0),
    ("Step 6", "EAF2.2", 78, 9.3, 0.412552, 1, 0, 0),
    ("Step 6", "EAF2.3", 80, 9.4, 0.000000, 1, 0, 0),
    ("Step 6", "EAF2.4", 76, 9.2, 0.825104, 1, 0, 0),

    ("Step 7", "Secondary Downstream 2.1", 28, 0.8, 32.179056, 0, 1, 0),
    ("Step 7", "Secondary Downstream 2.2", 24, 0.6, 30.941400, 0, 1, 0),
    ("Step 7", "Secondary Downstream 2.3", 20, 0.4, 15.470700, 0, 1, 0),
]

df_sequence = pd.DataFrame(
    stage_sequence,
    columns=[
        "Step", "Production Phase", "Total Load (MWh)",
        "Recovered Heat (MWh)", "Heat Required (MWh)",
        "EAF", "2ndary", "critical load MW"
    ]
)
df_sequence["t_in_day"] = range(1, 25)

# =========================================================
# 3. HELPER: DETECT COLUMNS
# =========================================================
def detect_column(df, keywords, exclude=None):
    exclude = exclude or []
    for col in df.columns:
        cl = str(col).strip().lower()
        if col in exclude:
            continue
        if all(k in cl for k in keywords):
            return col
    for col in df.columns:
        cl = str(col).strip().lower()
        if col in exclude:
            continue
        if any(k in cl for k in keywords):
            return col
    return None

# =========================================================
# 4. READ PRICE / GAS / SOLAR FILE
# =========================================================
raw = pd.read_excel(input_prices_file)

if raw.empty:
    raise ValueError("The prices file is empty.")

raw.columns = [str(c).strip() for c in raw.columns]

# Datetime
datetime_col = detect_column(raw, ["timestamp", "date", "time"])
if datetime_col is None:
    # Fallback if specific datetime column names are not found
    for col in raw.columns:
        # Exclude known price/availability columns from datetime detection fallback
        if any(k in str(col).lower() for k in ["electric", "price", "gas", "natural", "solar", "availability"]):
            continue
        parsed = pd.to_datetime(raw[col], errors="coerce")
        if parsed.notna().sum() > 100:
            datetime_col = col
            break

if datetime_col is None:
    raise ValueError("Could not detect datetime column.")

# Electricity price
elec_col = detect_column(raw, ["electric"])
if elec_col is None:
    elec_col = detect_column(raw, ["price"], exclude=[datetime_col])

# Natural gas
gas_col = detect_column(raw, ["gas"])
if gas_col is None:
    gas_col = detect_column(raw, ["natural"])

# Solar availability
solar_col = detect_column(raw, ["solar"])
if solar_col is None:
    solar_col = detect_column(raw, ["availability"])

if elec_col is None:
    raise ValueError("Could not detect electricity price column.")
if gas_col is None:
    raise ValueError("Could not detect natural gas price column.")
if solar_col is None:
    raise ValueError("Could not detect solar availability column.")

df = raw[[datetime_col, elec_col, gas_col, solar_col]].copy()
df.columns = ["Datetime", "Electricity Price", "Natural Gas Price", "Solar Availability"]
# Drop rows where Datetime or Electricity Price are missing BEFORE type conversion for robustness
df = df.dropna(subset=["Datetime", "Electricity Price"]).copy()

df["Datetime"] = pd.to_datetime(df["Datetime"])
df["Electricity Price"] = pd.to_numeric(df["Electricity Price"], errors="coerce")
df["Natural Gas Price"] = pd.to_numeric(df["Natural Gas Price"], errors="coerce")
df["Solar Availability"] = pd.to_numeric(df["Solar Availability"], errors="coerce")
df = df.dropna(subset=["Electricity Price"]).sort_values("Datetime").reset_index(drop=True)

df["Date"] = df["Datetime"].dt.date
df["Hour"] = df["Datetime"].dt.hour

# Keep only complete days
counts = df.groupby("Date").size()
complete_days = counts[counts == 24].index
df = df[df["Date"].isin(complete_days)].copy().reset_index(drop=True)

# =========================================================
# 5. FIND LOWEST 8 CONSECUTIVE DAYS
# =========================================================
daily_avg = (
    df.groupby("Date", as_index=False)["Electricity Price"]
      .mean()
      .rename(columns={"Electricity Price": "Daily Average Electricity Price"})
)
daily_avg["Date"] = pd.to_datetime(daily_avg["Date"])
daily_avg = daily_avg.sort_values("Date").reset_index(drop=True)

window_rows = []
best_avg = None
best_start_idx = None

for i in range(len(daily_avg) - 8 + 1):
    w = daily_avg.iloc[i:i+8].copy()
    start_date = w["Date"].iloc[0]
    end_date = w["Date"].iloc[-1]
    expected = pd.date_range(start=start_date, periods=8, freq="D")
    if list(w["Date"]) != list(expected):
        continue

    avg_val = w["Daily Average Electricity Price"].mean()
    window_rows.append({
        "Window Start": start_date,
        "Window End": end_date,
        "Average Electricity Price Over 8 Days": avg_val
    })

    if best_avg is None or avg_val < best_avg:
        best_avg = avg_val
        best_start_idx = i

if best_start_idx is None:
    raise ValueError("No valid 8-day consecutive window found.")

df_window_ranking = pd.DataFrame(window_rows).sort_values(
    "Average Electricity Price Over 8 Days"
).reset_index(drop=True)

best_start_date = daily_avg.iloc[best_start_idx]["Date"]
best_end_date = best_start_date + pd.Timedelta(days=7)

selected = df[
    (pd.to_datetime(df["Date"]) >= best_start_date) &
    (pd.to_datetime(df["Date"]) <= best_end_date)
].copy().sort_values("Datetime").reset_index(drop=True)

if len(selected) != 192:
    raise ValueError(f"Expected 192 rows for selected 8-day window, found {len(selected)}.")

# =========================================================
# 6. DAILY BATTERY / THERMAL DECISION RULE
# =========================================================
def choose_binary_controls(day_sched):
    """
    Heuristic control selection per day.
    Goal:
      - Battery Charge count >= 4
      - Battery Discharge count <= 3
      - Thermal Discharge count < 9
      - Prefer charging in low-price / solar-rich hours
      - Prefer discharging in high-price hours
      - Prefer thermal discharge in high heat-required hours
    """
    n = len(day_sched)
    day_sched = day_sched.copy()

    # initialize
    day_sched["Battery Charge"] = 0
    day_sched["Battery Discharge"] = 0
    day_sched["Thermal Discharge"] = 0

    # Battery charge: choose 4 hours among lower prices and good solar
    charge_score = (
        day_sched["Solar Availability"].fillna(0) * 10
        - day_sched["Electricity Price"]
    )
    charge_idx = charge_score.nlargest(4).index
    day_sched.loc[charge_idx, "Battery Charge"] = 1

    # Battery discharge: choose up to 3 highest-price hours not already charging
    discharge_candidates = day_sched[day_sched["Battery Charge"] == 0]
    discharge_idx = discharge_candidates["Electricity Price"].nlargest(3).index
    day_sched.loc[discharge_idx, "Battery Discharge"] = 1

    # Thermal discharge: choose up to 8 highest heat-required hours
    thermal_idx = day_sched["Heat Required (MWh)"].nlargest(8).index
    day_sched.loc[thermal_idx, "Thermal Discharge"] = 1

    return day_sched

# =========================================================
# 7. BUILD DAILY SCHEDULE FOR A GIVEN START HOUR
# =========================================================
def build_day_schedule(day_df, start_hour):
    """
    Daily 24-process circular schedule.
    """
    day_df = day_df.sort_values("Hour").reset_index(drop=True)
    if len(day_df) != 24:
        raise ValueError("Each daily set must contain exactly 24 hours.")

    rows = []
    for i, seq_row in df_sequence.iterrows():
        idx = (start_hour + i) % 24
        market_row = day_df.iloc[idx]

        rows.append({
            "t_in_day": int(seq_row["t_in_day"]),
            "Step": seq_row["Step"],
            "Production Phase": seq_row["Production Phase"],
            "Assigned Datetime": market_row["Datetime"],
            "Assigned Hour": int(market_row["Hour"]),
            "Electricity Price": float(market_row["Electricity Price"]),
            "Natural Gas Price": float(market_row["Natural Gas Price"]) if pd.notna(market_row["Natural Gas Price"]) else np.nan,
            "Solar Availability": float(market_row["Solar Availability"]) if pd.notna(market_row["Solar Availability"]) else 0.0,
            "Total Load (MWh)": float(seq_row["Total Load (MWh)"]),
            "Recovered Heat (MWh)": float(seq_row["Recovered Heat (MWh)"]),
            "Heat Required (MWh)": float(seq_row["Heat Required (MWh)"]),
            "ΕAF": int(seq_row["EAF"]),
            "EAF,binary": int(seq_row["EAF"]),
            "2ndary": int(seq_row["2ndary"]),
            "critical load MW": float(seq_row["critical load MW"]),
        })

    sched = pd.DataFrame(rows)

    # PV generation
    sched["PV Generation MWh"] = (
        sched["Solar Availability"].fillna(0) * PV_EFFICIENCY * PV_AREA_M2 / 1000.0
    )

    # Baseline binary decisions
    sched = choose_binary_controls(sched)

    # constant rates
    sched["Battery Charge/Disch Rate Rb (fraction)"] = RB_FRACTION
    sched["Thermal Charge/Disch Rate Rth (fraction)"] = RTH_FRACTION

    # Battery energy / efficiency / thermal capacity
    sched["Battery energy MWh"] = BATTERY_ENERGY_MWH
    sched["Battery efficiency (Fraction)"] = BATTERY_EFFICIENCY
    sched["Thermal buffer capacity MWh_th"] = THERMAL_BUFFER_CAPACITY_MWH_TH

    # battery SOC
    battery_charge_energy = BATTERY_ENERGY_MWH * RB_FRACTION
    battery_discharge_energy = BATTERY_ENERGY_MWH * RB_FRACTION

    soc = []
    current_soc = BATTERY_INITIAL_SOC
    for _, r in sched.iterrows():
        if r["Battery Charge"] == 1 and r["Battery Discharge"] == 0:
            current_soc = min(
                BATTERY_ENERGY_MWH,
                current_soc + battery_charge_energy * BATTERY_EFFICIENCY
            )
        elif r["Battery Discharge"] == 1 and r["Battery Charge"] == 0:
            current_soc = max(
                0.0,
                current_soc - battery_discharge_energy / BATTERY_EFFICIENCY
            )
        soc.append(current_soc)

    sched["Battery SOC (MWh)"] = soc

    # cyclic correction to make SOC_t=1 = SOC_t=24 approximately
    soc_diff = sched["Battery SOC (MWh)"].iloc[-1] - sched["Battery SOC (MWh)"].iloc[0]
    if abs(soc_diff) > 1e-6:
        sched["Battery SOC (MWh)"] = sched["Battery SOC (MWh)"] - np.linspace(0, soc_diff, len(sched))

    # thermal SOC
    thermal_charge = sched["Recovered Heat (MWh)"].fillna(0).values
    thermal_discharge = sched["Thermal Discharge"].values * (THERMAL_BUFFER_CAPACITY_MWH_TH * RTH_FRACTION)

    th_soc = []
    current_th_soc = THERMAL_INITIAL_SOC
    for i in range(len(sched)):
        current_th_soc = min(
            THERMAL_BUFFER_CAPACITY_MWH_TH,
            max(
                0.0,
                current_th_soc + thermal_charge[i] - thermal_discharge[i]
            )
        )
        th_soc.append(current_th_soc)

    sched["Thermal_SOC"] = th_soc

    th_diff = sched["Thermal_SOC"].iloc[-1] - sched["Thermal_SOC"].iloc[0]
    if abs(th_diff) > 1e-6:
        sched["Thermal_SOC"] = sched["Thermal_SOC"] - np.linspace(0, th_diff, len(sched))
        sched["Thermal_SOC"] = sched["Thermal_SOC"].clip(lower=0, upper=THERMAL_BUFFER_CAPACITY_MWH_TH)

    # grid import
    batt_discharge_mwh = sched["Battery Discharge"] * battery_discharge_energy
    batt_charge_mwh = sched["Battery Charge"] * battery_charge_energy
    thermal_offset = sched["Thermal Discharge"] * (THERMAL_BUFFER_CAPACITY_MWH_TH * RTH_FRACTION)

    sched["Grid_import"] = (
        sched["Total Load (MWh)"]
        + batt_charge_mwh
        - sched["PV Generation MWh"]
        - batt_discharge_mwh
    ).clip(lower=0)

    # scrap rate: simple placeholder linked to EAF and higher grid dependency
    sched["Scrap_rate"] = SCRAP_RATE_DEFAULT + 0.001 * sched["ΕAF"] * (sched["Grid_import"] / sched["Total Load (MWh)"].replace(0, 1))

    # costs
    sched["Total Cost without PV and Heat Recover"] = (
        sched["Total Load (MWh)"] * sched["Electricity Price"]
    )

    sched["Total Cost with PV and Storage"] = (
        sched["Grid_import"] * sched["Electricity Price"]
    )

    thermal_gas_needed = (
        sched["Heat Required (MWh)"] - sched["Recovered Heat (MWh)"] - thermal_offset
    ).clip(lower=0)

    sched["Total Cost with PV+Storage+Scrap+Thermal"] = (
        sched["Grid_import"] * sched["Electricity Price"]
        + thermal_gas_needed * sched["Natural Gas Price"].fillna(0) * THERMAL_TO_GAS_CONVERSION
        + sched["Scrap_rate"] * sched["Total Load (MWh)"] * sched["Electricity Price"] * 0.10
    )

    daily_cost = sched["Total Cost with PV+Storage+Scrap+Thermal"].sum()

    return sched, daily_cost

# =========================================================
# 8. OPTIMIZE EACH OF THE 8 DAYS
# =========================================================
daily_optimization_rows = []
optimized_rows = []
global_t = 1
total_cost_8d = 0.0

selected_dates = sorted(selected["Date"].unique())

for d in selected_dates:
    day_df = selected[selected["Date"] == d].copy()
    if len(day_df) != 24:
        raise ValueError(f"Date {d} does not have 24 rows.")

    best_start = None
    best_cost = None
    best_sched = None

    for start_hour in range(24):
        sched, cost = build_day_schedule(day_df, start_hour)
        if best_cost is None or cost < best_cost:
            best_cost = cost
            best_start = start_hour
            best_sched = sched.copy()

    total_cost_8d += best_cost

    daily_optimization_rows.append({
        "Date": pd.Timestamp(d),
        "Optimal Start Hour (EAF1.1)": best_start,
        "Minimum Daily Cost with PV+Storage+Scrap+Thermal": best_cost
    })

    for _, r in best_sched.iterrows():
        rec = r.to_dict()
        rec["Global t"] = global_t
        rec["Date"] = pd.Timestamp(d)
        optimized_rows.append(rec)
        global_t += 1

df_daily_optimization = pd.DataFrame(daily_optimization_rows)
df_optimized_schedule = pd.DataFrame(optimized_rows)

# Reorder columns
ordered_cols = [
    "Global t", "Date", "t_in_day", "Assigned Datetime", "Assigned Hour",
    "Step", "Production Phase",
    "Electricity Price", "Natural Gas Price", "Solar Availability",
    "Battery energy MWh", "Battery efficiency (Fraction)", "Thermal buffer capacity MWh_th",
    "PV Generation MWh", "ΕAF", "EAF,binary", "2ndary", "critical load MW",
    "Battery Charge", "Battery Discharge", "Thermal Discharge",
    "Battery Charge/Disch Rate Rb (fraction)", "Battery SOC (MWh)",
    "Thermal Charge/Disch Rate Rth (fraction)", "Thermal_SOC",
    "Grid_import", "Scrap_rate",
    "Total Load (MWh)", "Recovered Heat (MWh)", "Heat Required (MWh)",
    "Total Cost without PV and Heat Recover",
    "Total Cost with PV and Storage",
    "Total Cost with PV+Storage+Scrap+Thermal"
]
df_optimized_schedule = df_optimized_schedule[ordered_cols]

# =========================================================
# 9. SUMMARY + NOTES
# =========================================================
df_summary = pd.DataFrame({
    "Metric": [
        "Selected 8-day window start",
        "Selected 8-day window end",
        "Lowest 8-day average electricity price",
        "Optimized days",
        "Total process times",
        "Total optimized cost with PV + storage + scrap + thermal"
    ],
    "Value": [
        best_start_date,
        best_end_date,
        best_avg,
        8,
        192,
        total_cost_8d
    ]
})

df_notes = pd.DataFrame({
    "Notes": [
        "The script selects the 8 consecutive days with the lowest average electricity price.",
        "The corresponding natural gas prices and solar availability values are taken for the same timestamps.",
        "Each day is optimized independently as one 24-process circle.",
        "PV Generation MWh = Solar Availability * 0.18 * 66700 / 1000.",
        "Battery Charge, Battery Discharge and Thermal Discharge are generated by a heuristic daily minimization rule.",
        "A constant daily battery rate Rb and thermal rate Rth are used.",
        "SOC columns are corrected to satisfy approximately SOC_t=1 = SOC_t=24 for each 24-hour daily circle.",
        "If your Design.xlsx includes additional exact formulas, replace the formula block in this script with those expressions."
    ]
})

# =========================================================
# 10. WRITE EXCEL
# =========================================================
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df_summary.to_excel(writer, sheet_name="Summary", index=False)
    df_window_ranking.to_excel(writer, sheet_name="8-Day Window Ranking", index=False)
    selected[["Datetime", "Date", "Hour", "Electricity Price", "Natural Gas Price", "Solar Availability"]].to_excel(
        writer, sheet_name="Selected Prices", index=False
    )
    df_daily_optimization.to_excel(writer, sheet_name="Daily Optimization", index=False)
    df_optimized_schedule.to_excel(writer, sheet_name="Optimized Schedule", index=False)
    df_notes.to_excel(writer, sheet_name="Notes", index=False)

# =========================================================
# 11. FORMAT EXCEL
# =========================================================
wb = load_workbook(output_file)

header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)
thin = Side(style="thin", color="BFBFBF")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

phase_fills = {
    "EAF": PatternFill("solid", fgColor="FCE4D6"),
    "Secondary": PatternFill("solid", fgColor="E2F0D9"),
    "SD sequence Blue Colour": PatternFill("solid", fgColor="D9EAF7"),
    "Minimum Critical Load": PatternFill("solid", fgColor="F4CCCC"),
    "Preparation Blue": PatternFill("solid", fgColor="D9E1F2"),
}

for ws in wb.worksheets:
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = border

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.border = border
            cell.alignment = Alignment(vertical="center")

    for col_cells in ws.columns:
        max_len = 0
        col_letter = col_cells[0].column_letter
        for cell in col_cells:
            v = "" if cell.value is None else str(cell.value)
            max_len = max(max_len, len(v))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 30)

for sheet_name in ["Summary", "8-Day Window Ranking", "Selected Prices", "Daily Optimization", "Optimized Schedule"]:
    ws = wb[sheet_name]
    headers = [c.value for c in ws[1]]
    for col_idx, header in enumerate(headers, start=1):
        if header and any(k in str(header) for k in ["Price", "Cost", "SOC", "Rate", "Load", "Generation", "import", "Scrap", "Heat"]):
            for row in range(2, ws.max_row + 1):
                ws.cell(row=row, column=col_idx).number_format = "0.00"

ws_opt = wb["Optimized Schedule"]
headers_opt = [c.value for c in ws_opt[1]]
phase_col = headers_opt.index("Production Phase") + 1

for row in range(2, ws_opt.max_row + 1):
    phase = str(ws_opt.cell(row=row, column=phase_col).value)
    fill = None
    if phase.startswith("EAF"):
        fill = phase_fills["EAF"]
    elif phase.startswith("Secondary"):
        fill = phase_fills["Secondary"]
    elif phase == "SD sequence Blue Colour":
        fill = phase_fills["SD sequence Blue Colour"]
    elif phase.startswith("Minimum Critical Load"):
        fill = phase_fills["Minimum Critical Load"]
    elif phase.startswith("Preparation Blue"):
        fill = phase_fills["Preparation Blue"]
    if fill:
        ws_opt.cell(row=row, column=phase_col).fill = fill

wb.save(output_file)

# =========================================================
# 12. PRINT RESULTS
# =========================================================
print(f"Output saved to: {output_file}")
print(f"Selected 8-day window: {best_start_date} to {best_end_date}")
print(f"Lowest 8-day average electricity price: {best_avg:.2f}")
print(f"Total optimized cost with PV + storage + scrap + thermal: {total_cost_8d:,.2f}")
print("\nDaily optimal start hours:")
for _, row in df_daily_optimization.iterrows():
    print(f"{row['Date'].date()} -> {int(row['Optimal Start Hour (EAF1.1)'])}")

Output saved to: Lowest_8Day_Mini_Mill_with_Gas_Solar_PV_Storage.xlsx
Selected 8-day window: 2025-04-27 00:00:00 to 2025-05-04 00:00:00
Lowest 8-day average electricity price: 54.10
Total optimized cost with PV + storage + scrap + thermal: 254,850.99

Daily optimal start hours:
2025-04-27 -> 12
2025-04-28 -> 12
2025-04-29 -> 12
2025-04-30 -> 11
2025-05-01 -> 12
2025-05-02 -> 12
2025-05-03 -> 12
2025-05-04 -> 11
